In [1]:
"""
cv_final.py: 5-fold cross-validation of the final model, to report a reliable
F1 (mean +- std) in the report.

The two parts come from the shared files, so everything stays consistent:
  * validation.py -> folds (stratified, same as the rest of the group),
                     threshold choice and metrics (cross_validate)
  * preprocessing.py -> features (build_features + drop_duplicate_columns, as in run.py)
  * run.py        -> model (train, predict_proba),
                     i.e. exactly the pipeline of the submission

In each fold, everything is learned on the training folds only: cleaning
rules, one-hot categories, standardization, weights and the threshold
(threshold="inner": chosen on an inner split, then the model is retrained
on all the training folds, as in run.py).

Usage: python cv_final.py   (about 5 minutes)
"""
import numpy as np

from preprocessing import build_features, drop_duplicate_columns
from run import load_data, train, predict_proba
from validation import cross_validate, print_cv_summary

K = 5
SEED = 0


def main():
    x_train, _, y01, _, names = load_data()

    def preprocess(x_tr, x_ev):
        """Features of run.py, learned on x_tr and applied to x_ev."""
        return drop_duplicate_columns(*build_features(x_tr, x_ev, names))

    def fit_predict(x_tr, y_tr, x_ev, seed):
        """Model of run.py: train on (x_tr, y_tr), return probabilities on x_ev."""
        w, mean, std = train(x_tr, y_tr.astype(float))
        return predict_proba(x_ev, w, mean, std)

    result = cross_validate(x_train, y01, fit_predict, n_splits=K, seed=SEED,
                            preprocess=preprocess, threshold="inner", verbose=True)
    print()
    print_cv_summary(result)
    print(f"Mean accuracy: {result['mean']['accuracy']:.4f}; "
          f"std: {result['std']['accuracy']:.4f}")


if __name__ == "__main__":
    main()

Fold 1/5: F1=0.4203, threshold=0.202793


Fold 2/5: F1=0.4306, threshold=0.199051


Fold 3/5: F1=0.4263, threshold=0.211278


Fold 4/5: F1=0.4285, threshold=0.175728


Fold 5/5: F1=0.4284, threshold=0.211846



Fold  F1       Precision  Recall    Accuracy  Threshold
  1   0.4203   0.3476     0.5315    0.8706    0.202793
  2   0.4306   0.3531     0.5519    0.8712    0.199051
  3   0.4263   0.3660     0.5103    0.8787    0.211278
  4   0.4285   0.3330     0.6007    0.8585    0.175728
  5   0.4284   0.3614     0.5260    0.8761    0.211846
Mean F1: 0.4268; std: 0.0035
Aggregated out-of-fold F1: 0.4269
Mean accuracy: 0.8710; std: 0.0069
